# 02 Turnaround Case Construction

This notebook enriches the cleaned flights with aircraft, airport, and airline reference data, then pairs inbound and outbound legs into turnaround cases.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.build_turnaround_cases import build_turnaround_cases_df
from src.config import get_data_paths, load_config
from src.enrich_aircraft import enrich_with_aircraft
from src.enrich_airlines import enrich_with_airlines
from src.enrich_airports import enrich_with_airports
from src.utils import read_table, setup_logging, write_table

sns.set_theme(style='whitegrid')


In [ ]:
config = load_config(PROJECT_ROOT / 'configs' / 'default_config.yaml')
paths = get_data_paths(config, PROJECT_ROOT)
logger = setup_logging(config['logging']['level'])

_ = enrich_with_aircraft(PROJECT_ROOT / 'configs' / 'default_config.yaml')
_ = enrich_with_airports(PROJECT_ROOT / 'configs' / 'default_config.yaml')
_ = enrich_with_airlines(PROJECT_ROOT / 'configs' / 'default_config.yaml')

flights_enriched = read_table(paths['interim_dir'] / 'flights_enriched.parquet')
print(f'Enriched flight rows: {len(flights_enriched):,}')
flights_enriched.head()


In [ ]:
turnaround_cfg = config['processing']['turnaround']
turnaround_df = build_turnaround_cases_df(
    flights_enriched,
    min_minutes=int(turnaround_cfg['min_minutes']),
    max_minutes=int(turnaround_cfg['max_minutes']),
    logger=logger,
)
write_table(turnaround_df, paths['processed_dir'] / 'turnaround_cases.parquet')
print(f'Turnaround cases: {len(turnaround_df):,}')
turnaround_df.head()


In [ ]:
turnaround_df['turnaround_minutes'].describe(percentiles=[0.25, 0.5, 0.75, 0.9])


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(turnaround_df['turnaround_minutes'], bins=30, kde=True)
plt.title('Turnaround Duration Distribution')
plt.xlabel('Turnaround minutes')
plt.show()


In [ ]:
turnaround_df[['case_id', 'icao24', 'airport_icao', 'arrival_time', 'departure_time', 'turnaround_minutes']].sample(min(10, len(turnaround_df)), random_state=42)
